# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields using their @id
print("Available record sets (by @id):\n")
record_set_ids = []
for rs in dataset.record_sets():
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    record_set_ids.append(rs['@id'])

# For each record set, list fields with their @id and name
all_fields = {}
for rs in dataset.record_sets():
    print(f"\nFields for RecordSet @id={rs['@id']}:")
    all_fields[rs['@id']] = []
    for field in rs.get('field', []):
        field_id = field['@id'] if isinstance(field, dict) else field
        field_obj = dataset.schema.field(field_id)
        print(f"  - @id: {field_obj['@id']}, name: {field_obj.get('name', 'N/A')}, dataType: {field_obj.get('dataType', 'N/A')}")
        all_fields[rs['@id']].append(field_obj['@id'])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
print("\nLoading data into DataFrames by RecordSet @id:\n")
for record_set_id in record_set_ids:
    print(f"  Loading RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"    Columns: {df.columns.tolist()}")
    print(f"    Example rows: \n{df.head(2)}\n")

# Pick the first record set for subsequent analysis
primary_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else Noneif primary_record_set_id:
    print(f"Primary record set ID to use: {primary_record_set_id}")
    df = dataframes[primary_record_set_id]
    print("Columns:", df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on a numeric field
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Find a numeric (float or integer) field from the schema
    num_field_id = None
    group_field_id = None
    # Use schema to find metadata about fields
    fields = dataset.schema.record_set(primary_record_set_id)['field']
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        field_obj = dataset.schema.field(field_id)
        dtype = str(field_obj.get('dataType','')).lower()
        if (num_field_id is None) and (('float' in dtype) or ('integer' in dtype) or ('number' in dtype) or ('double' in dtype)):
            num_field_id = field_obj['@id']
        if (group_field_id is None) and (('name' in field_obj.get('name','').lower()) or ('accession' in field_obj.get('name','').lower())) and ('string' in dtype):
            group_field_id = field_obj['@id']

    # If can't find, just use first column as fallback
    if not num_field_id:
        # Try to guess from DataFrame columns
        for c in df.columns:
            if np.issubdtype(df[c].dropna().infer_objects().dtype, np.number):
                num_field_id = c
                break
    if num_field_id and num_field_id in df.columns:
        # Try to convert to numeric
        df[num_field_id] = pd.to_numeric(df[num_field_id], errors='coerce')
        threshold = np.nanpercentile(df[num_field_id], 80)  # Use 80th percentile for sample threshold
        filtered_df = df[df[num_field_id] > threshold].copy()
        print(f"Filtered records with {num_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        norm_col = f"{num_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[num_field_id] - filtered_df[num_field_id].mean()) / filtered_df[num_field_id].std()
        print(f"\nNormalized {num_field_id} for filtered records:")
        print(filtered_df[[num_field_id, norm_col]].head())

        # Group by group_field if available
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id, dropna=False).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (mean):")
            print(grouped_df.head())
        else:
            print("\nNo group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and num_field_id:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[num_field_id].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {num_field_id}")
    ax.set_xlabel(num_field_id)
    plt.show()

    # Scatter plot if possible
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sample = df.dropna(subset=[num_field_id,group_field_id]).groupby(group_field_id).mean(numeric_only=True).reset_index()
        sns.barplot(x=group_field_id, y=num_field_id, data=sample)
        plt.title(f"Mean {num_field_id} per {group_field_id}")
        plt.ylabel(f"Mean {num_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to explore, filter, and visualize data from the FAIR² dataset titled "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library. We loaded metadata and data records using their Croissant schema `@id`, explored the available fields, performed normalization and grouping of numeric data, and visualized distributions. For more advanced analyses, you can tailor the EDA and visualizations to your specific scientific queries.